two things
# 1. spark optimization techniques
# 2. delta optimization techniques

In [0]:
from pyspark.sql.functions import col
df= spark.table("formula1_dev.bronze.circuits")
df1=df.filter(col("circuit_id")==71)\
    .select(col("circuit_ref"),col("circuit_id"))\
    .orderBy(col("circuit_id"))

In [0]:
# df1.explain("formatted")
df1.explain()

In [0]:
from 
join
filter
group 
having
select 
order 
limit 

In [0]:
predicate pushdown ==> early filter 
column pruning ==> select specific columns


In [0]:
shuffle --> data across the partitions [spark]
to reduce --> partitions of the data should be reduced. 
to read the files --> process the data .. shuffle will be more 
low cardinality columns --> reduce the shuffle --> country , location ,region 
# partitionBy("country")

In [0]:
executors --> work chesthundi --> plan -->transformations
driver --> results 

In [0]:
DAG --> direct/directed acyclic graph 
when you code --> run
directed --> every edge has a direction data flow from one operation to another operation [parent --> child ]
-->*-->*
Acyclic --> no cycles --> no loops --> no circular dependencies 
Graph --> collection of nodes and eges 

In [0]:
spark builds the dag lazily --> when you run the code like filter ,joins , groupby
spark dag lazy build execution bluepront for your job 

In [0]:
transformations -->  narrow and wide transformations 
every transformation filter, group by --> just adds a node to the logical dag 
spark tracks two kinds of dependency between nodes:
    1. Narrow dependency/narrow transformation --> parent and child partitions are in the same executor
    parent--> child partitions are in the same executor 
    2.wide dependency/wide transformation --> parent and child partitions are in different executors
    parent -> multiple child partitions 
1 partition --> multiple partitions create avuthundi

In [0]:
print(spark.version)

In [0]:
3. things for aqe
1. spark.sql.shuffle.partitions = 200[default] this creates numbr of partitions at the time of the shuffling 
spark.sql.maxPartitionBytes = 128 MB this is the max size of the partition

In [0]:
spark.conf.set("spark.sql.maxPartitionBytes", 512) # file reading partition size 
spark.conf.set("spark.sql.shuffle.partitions", 250) # shuffling the data across the executors transfomations like joins, group by 

In [0]:
df.write.mode("delta").partitionBy("state").saveAsTable("table_name")



In [0]:
10 MB --> partition size of the data in the memory
10 MB --> 10MB 

In [0]:
results_df = spark.table("formula1_dev.silver.results")
races_df = spark.table("formula1_dev.silver.races")
drivers_df = spark.table("formula1_dev.silver.drivers")



demo_races_df = races_df
demo_results_df = results_df

print("Demo races:", demo_races_df.count())
print("Demo results:", demo_results_df.count())

In [0]:
baseline_df = (
    demo_results_df.alias("r")
    .join(
        demo_races_df.alias("ra"),
        F.col("r.race_id") == F.col("ra.race_id"),
        "inner"
    )
    .join(
        drivers_df.alias("d"),
        F.col("r.driver_id") == F.col("d.driver_id"),
        "inner"
    )
    .groupBy(
        "race_year",
        F.col("r.driver_id"),
        F.col("d.driver_full_name")
    )
    .agg(
        F.sum("r.points").alias("total_points"),
        F.countDistinct("r.race_id").alias("races_entered")
    )
)

baseline_df.orderBy(
    "race_year",
    F.col("total_points").desc()
).show(20, False)

In [0]:
from pyspark.sql.functions import *
shuffle_df = (
    demo_results_df
    .groupBy("driver_id")
    .agg(sum("points").alias("total_points"))
)

shuffle_df.show(10, False)
shuffle_df.explain(True)

In [0]:
# results_df.count() 25000
# races_df.count() 1058

In [0]:
# 2. broadcast join
from pyspark.sql.functions import * 
broadcast_join_df = demo_results_df.alias("r").join(
    broadcast(demo_races_df).alias("ra"),
    col("r.race_id") == col("ra.race_id"), "inner")
broadcast_join_df.count()

In [0]:
broadcast_join_df.explain()

In [0]:
# 2. broadcast join
from pyspark.sql.functions import * 
broadcast_join_df1 = demo_results_df.alias("r").join(
    demo_races_df.alias("ra"),
    col("r.race_id") == col("ra.race_id"), "inner")
broadcast_join_df1.count()

In [0]:
broadcast_join_df1.explain()

In [0]:
# improper data distribution across partitions - skew data
# shuffle is data distrubtion across pratitions 

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1) # no broad cast join threshold

In [0]:
drivers_df.count()

In [0]:
#sort merge join 

results_df.alias("r").join(drivers_df.alias("d"), col("r.driver_id") == col("d.driver_id"), "inner").explain()

In [0]:
shuffle_hash_df1 = results_df.alias("r").join(drivers_df.alias("d"), col("r.driver_id") == col("d.driver_id"), "inner").hint("shuffle_hash")
shuffle_hash_df1.explain("formatted")

In [0]:
external and internal tables 
path == > external 
managed/internal table ---> no path 

In [0]:
cache /persist / unpersist 
coalesce / repartion
vaccum 


In [0]:
df.write.mode("overwrite").option(mergeSchema, "True").saveAsTable("table_name") # schema evolution # schema enforcement
df.write.mode("overwrite").format("delta").saveAsTable("table_name")  # schema enforcement 